# Module 15 — Decorators, Closures, and functools

## Exercise 15.3 — Cache traps, measured

Each section demonstrates a real bug. Predict, run, then explain.
Run:  python ex03_functools.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. `functools.wraps` is not optional

In [ ]:
@log_calls
def add(a, b):
    """Add two numbers."""

add.__name__      # 'wrapper'    <- wrong
add.__doc__       # None         <- gone
inspect.signature(add)   # (*args, **kwargs)   <- useless

The wrapper replaced the function, so all of its metadata is the wrapper's.
What breaks, concretely:

- `help()` and every documentation generator
- debuggers and profilers reporting "wrapper" for every decorated function
- **pytest fixture resolution**, which inspects parameter names
- **FastAPI and Pydantic**, which build schemas from signatures
- `singledispatch`, which reads annotations
- any logging that uses `__name__`

In [ ]:
import functools

def log_calls(fn):
    @functools.wraps(fn)          # copies __name__, __doc__, __module__,
    def wrapper(*args, **kwargs): # __qualname__, __dict__, and sets __wrapped__
        return fn(*args, **kwargs)
    return wrapper

`__wrapped__` is what lets `inspect.signature` see through the wrapper to the
real signature. **Always use `wraps`.** There is no case where omitting it is
correct.

---

## Concept 3. Decorators with arguments: three levels

In [ ]:
def retry(attempts=3, delay=1.0):        # 1. the FACTORY takes the arguments
    def decorator(fn):                   # 2. the DECORATOR takes the function
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):    # 3. the WRAPPER takes the call
            for attempt in range(attempts):
                try:
                    return fn(*args, **kwargs)
                except Exception:
                    if attempt == attempts - 1:
                        raise
                    time.sleep(delay)
        return wrapper
    return decorator

@retry(attempts=5)          # note: CALLED. retry(5) returns `decorator`.
def flaky(): ...

`@retry` without parentheses passes the *function* as `attempts`, and the error
appears far away and makes no sense. To support both forms:

```text
def retry(fn=None, *, attempts=3):
    if fn is None:                       # called with arguments
        return functools.partial(retry, attempts=attempts)
    @functools.wraps(fn)
    def wrapper(*a, **kw): ...
    return wrapper

@retry              # works
@retry(attempts=5)  # also works
```


---

## Concept 4. Stacking order

In [ ]:
@a
@b
@c
def f(): ...

# f = a(b(c(f)))

**Bottom-up at definition; top-down at call time.** The decorator closest to the
`def` wraps first, so it is *innermost*, so its wrapper code runs *last* on the
way in.

This ordering matters and gets people:

In [ ]:
@app.route("/admin")        # registers whatever is beneath it
@require_auth               # so the route registered is the AUTHENTICATED one
def admin(): ...

@require_auth               # WRONG ORDER
@app.route("/admin")        # registers the RAW function; auth is never applied
def admin(): ...

The second version registers the undecorated function with the framework and
then wraps a name nobody calls. It looks right, runs fine, and has no
authentication.

Rules of thumb: `@staticmethod`/`@classmethod` outermost; caching outside
logging (so cached calls are not logged as work); registration outermost so it
registers the fully decorated function.

---

## Concept 5. `functools`

### `lru_cache` / `cache`

In [ ]:
@functools.lru_cache(maxsize=128)
def expensive(n: int) -> int: ...

@functools.cache                  # 3.9+: unbounded lru_cache
def fib(n: int) -> int:
    return n if n < 2 else fib(n - 1) + fib(n - 2)

expensive.cache_info()            # hits, misses, maxsize, currsize
expensive.cache_clear()

Five things to know before using it:

1. **Arguments must be hashable.** A list argument raises `TypeError`.
2. **Equal-but-distinct arguments can collide.** `1`, `1.0` and `True` are equal
   and hash equally (Module 03), so a function that treats them differently can
   get the wrong cached answer. The exact behaviour is subtler than it looks —
   `lru_cache` has a fast path for a single `int` or `str` argument, so the real
   grouping is not the one you would predict. Exercise 15.3 measures it. The
   safe rule: if your function's behaviour depends on the *type* of a numeric
   argument, do not cache it by that argument.
3. **`f(1)` and `f(x=1)` are different entries.** Same call, two cache slots.
4. **It keeps a strong reference to every argument and result.** `@cache` on a
   method keeps every instance alive forever — a genuine and common memory leak.
   Use `maxsize`, or `cached_property`, or a `WeakValueDictionary`.
5. **Only cache pure functions.** A cached function with side effects performs
   them once and silently skips them thereafter.

### `partial`

In [ ]:
from functools import partial
int2 = partial(int, base=2)
int2("1010")                       # 10
sorted(rows, key=partial(get_field, "name"))

`partial` beats a lambda for a callback: it has a useful `repr`, it is
picklable (so it works with `multiprocessing`, Module 21), and it does not
capture variables by reference — which sidesteps Module 04's late-binding trap.

### `singledispatch`

In [ ]:
@functools.singledispatch
def serialise(obj) -> str:
    raise TypeError(f"cannot serialise {type(obj).__name__}")

@serialise.register
def _(obj: datetime) -> str: return obj.isoformat()

@serialise.register
def _(obj: Decimal) -> str: return str(obj)

Type-based dispatch without an `isinstance` chain, and — importantly —
**open for extension**: a third party can register a handler for their own type
without touching your code. This is the Visitor pattern, dissolved (Module 12).

`singledispatchmethod` does the same for methods.

### `cached_property`, `total_ordering`, `reduce`

In [ ]:
@functools.cached_property        # Module 08: computed once, stored in __dict__
def stats(self): ...

@functools.total_ordering         # Module 09: fills in <=, >, >= from < and ==
class Version: ...

functools.reduce(operator.mul, nums, 1)     # rarely clearer than a loop

`reduce` is worth knowing and rarely worth using. `sum`, `math.prod`,
`itertools.accumulate` and an explicit loop are all clearer.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: `@` is sugar
- Section 2: `functools.wraps` is not optional
- Section 3: Decorators with arguments: three levels
- Section 4: Stacking order
- Section 5: `functools`
- Section 6: `contextlib`
- Section 7: Class-based decorators, and when to use one

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import functools
import gc
import sys
import weakref
from typing import Any


# --- trap 1: the equal-but-different keys -------------------------------------

---

## `trap_equal_keys`

PREDICT: how many cache entries after f(1), f(1.0), f(True), f(n=1)?

In [ ]:
def trap_equal_keys() -> None:
    """PREDICT: how many cache entries after f(1), f(1.0), f(True), f(n=1)?

    The answer is not what the README's simple version implies, and finding out
    why is the exercise. Read functools._make_key -- it has a FAST PATH for a
    single argument of certain types. Which types, and what does that do to the
    grouping?

    Then answer the question that matters: write a function where this
    difference produces a WRONG ANSWER, not just a surprising cache size.
    """
    @functools.cache
    def f(n: Any) -> str:
        return f"{n!r} is {type(n).__name__}"

    print("trap 1")
    for call, label in [((1,), "f(1)"), ((1.0,), "f(1.0)"),
                        ((True,), "f(True)")]:
        print(f"    {label:<10} -> {f(*call)}")
    print(f"    f(n=1)     -> {f(n=1)}")
    print(f"    entries: {f.cache_info().currsize}   {f.cache_info()}")

---

## `Heavy`

Pretend each instance holds 10 MB.

In [ ]:
class Heavy:
    """Pretend each instance holds 10 MB."""

    def __init__(self, name: str) -> None:
        self.name = name
        self.payload = bytearray(1024 * 1024)     # 1 MB, to keep it quick

    @functools.cache                               # THE BUG
    def expensive(self, factor: int) -> int:
        return len(self.payload) * factor

---

## `trap_method_leak`

PREDICT: after creating and dropping 20 Heavy objects, how many are

In [ ]:
def trap_method_leak() -> None:
    """PREDICT: after creating and dropping 20 Heavy objects, how many are
    still alive?

    Then answer:
      - why does the cache keep them? What is in the cache KEY?
      - name three fixes, and say what each costs
      - which fix does the standard library provide for exactly this case?
    """
    print("\ntrap 2")
    refs = []
    for i in range(20):
        obj = Heavy(f"h{i}")
        obj.expensive(2)
        refs.append(weakref.ref(obj))
        del obj

    gc.collect()
    alive = sum(1 for r in refs if r() is not None)
    print(f"    created 20, dropped all references, still alive: {alive}")
    print(f"    cache: {Heavy.expensive.cache_info()}")

---

## `trap_unhashable`

PREDICT: what happens, and is failing the right behaviour?

In [ ]:
def trap_unhashable() -> None:
    """PREDICT: what happens, and is failing the right behaviour?

    Then implement TWO workarounds and say when each is right:
      (a) convert at the boundary (tuple(items)) -- what does the caller lose?
      (b) a custom key function -- what does that cost, and what can go wrong
          if the key is not injective?
    """
    print("\ntrap 3")

    @functools.cache
    def process(items: Any) -> int:
        return sum(items)

    try:
        process([1, 2, 3])
    except TypeError as exc:
        print(f"    TypeError: {exc}")
    print(f"    tuple works: {process((1, 2, 3))}")

---

## `trap_impure`

PREDICT: how many lines does the log contain after three calls?

In [ ]:
def trap_impure() -> None:
    """PREDICT: how many lines does the log contain after three calls?

    Then answer: this is obvious when the side effect is a print. Name three
    side effects that are NOT obvious in review, where @cache silently breaks
    the function.
    """
    print("\ntrap 4")
    log: list[str] = []

    @functools.cache
    def save(record_id: int) -> str:
        log.append(f"saved {record_id}")
        return f"receipt-{record_id}"

    for _ in range(3):
        save(1)
    print(f"    called 3 times, log has {len(log)} entries: {log}")

---

## `trap_maxsize`

Measure the memory of an unbounded cache over a high-cardinality key.

In [ ]:
def trap_maxsize() -> None:
    """Measure the memory of an unbounded cache over a high-cardinality key.

    Then answer:
      - what is the right maxsize for a function keyed by user id, in a service
        with 10 million users and 50,000 daily actives?
      - what happens to hit rate at maxsize=128 versus 100_000?
      - when is maxsize=None actually correct?
    """
    print("\ntrap 5")

    @functools.cache
    def by_id(user_id: int) -> str:
        return f"user-{user_id}" * 10

    for i in range(50_000):
        by_id(i)
    size = sys.getsizeof(by_id.__wrapped__)  # not the real answer -- find it
    print(f"    entries: {by_id.cache_info().currsize:,}")
    print("    now measure the ACTUAL memory with tracemalloc, not getsizeof")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    trap_equal_keys()
    trap_method_leak()
    trap_unhashable()
    trap_impure()
    trap_maxsize()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.